# 🎯 RS-LiDAR & LiDAR: Chấm Lại Điểm GenEval (Mask2Former Swin-S)
### Bảng Tổng Hợp 6 Dòng Chuẩn Khoa Học Cho Cả 3 Settings:
1. **SD v1.5 (DDIM-50, $\eta=0.0$)**: Vanilla LiDAR vs RS-LiDAR ($\sigma=1.0, M=4$)
2. **SD v1.5 (DDPM-100, $\eta=1.0$)**: Vanilla LiDAR vs RS-LiDAR ($\sigma=1.0, M=4$)
3. **SDXL 2.6B (DDPM-100, $\eta=1.0$)**: Vanilla LiDAR vs RS-LiDAR ($\sigma=1.0, M=4$)

---
### 📌 Cơ Chế Hoạt Động (Chỉ Cần Bấm Run All):
- **Gắn kết Google Drive cá nhân** (`/content/drive/MyDrive/RS-LiDAR`).
- **Cố định đường dẫn cứng đúng 6 thư mục thực nghiệm**.
- **Lưu checkpoint tức thì**: Chấm xong thư mục nào là lưu `geneval_summary.csv` và cập nhật `table2_master_6rows_summary.csv` ngay lập tức sau thư mục đó (không sợ mất công nếu bị ngắt kết nối giữa chừng).
- **Đọc trực tiếp `final_metrics.json`** để lấy: **ImageReward, CLIP-Score, HPS v2.1**.
- **Chấm lại GenEval** bằng mô hình chuẩn bài báo gốc **Mask2Former Swin-S COCO** (`facebook/mask2former-swin-small-coco-instance`, mAP ~52%).
- **Đầu ra**: Bảng tổng kết chuẩn khoa học gồm **đúng 6 dòng** so sánh trực diện giữa Vanilla LiDAR và RS-LiDAR trên cả 3 settings!


## 1. Cài Đặt Môi Trường & Kiểm Tra GPU

In [ ]:
# @title 🚀 Cài đặt thư viện đánh giá chuẩn khoa học
!nvidia-smi

import os, sys, glob, json, time, shutil
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

# Cài đặt transformers, timm và các phụ thuộc cần thiết cho Mask2Former Native
!pip install -q --upgrade transformers timm tabulate accelerate

import torch
from transformers import AutoImageProcessor, Mask2FormerForUniversalSegmentation, CLIPProcessor, CLIPModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n✅ Môi trường PyTorch sẵn sàng! Sử dụng thiết bị: {device}")
if device == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


## 2. Gắn Kết Google Drive Cá Nhân & Khởi Tạo Đường Dẫn Cứng

In [ ]:
# @title 📁 Gắn kết Google Drive cá nhân & Khởi tạo đường dẫn
from google.colab import drive
drive.mount('/content/drive')

# Tự động phát hiện chính xác đường dẫn Google Drive (MyDrive hoặc My Drive)
base_drive = None
for candidate in ['/content/drive/MyDrive', '/content/drive/My Drive', '/content/drive']:
    if os.path.exists(f"{candidate}/RS-LiDAR/Target_samples"):
        base_drive = candidate
        break

if base_drive is None:
    base_drive = '/content/drive/MyDrive' if os.path.exists('/content/drive/MyDrive') else '/content/drive/My Drive'

DRIVE_DIR = f"{base_drive}/RS-LiDAR"
TARGET_BASE = f"{DRIVE_DIR}/Target_samples"

assert os.path.exists(DRIVE_DIR), f"❌ Không tìm thấy thư mục RS-LiDAR tại: {DRIVE_DIR}! Vui lòng kiểm tra lại Google Drive."
assert os.path.exists(TARGET_BASE), f"❌ Không tìm thấy thư mục Target_samples tại: {TARGET_BASE}!"

print(f"✅ Đã kết nối thành công với Google Drive cá nhân!")
print(f"📁 Thư mục gốc: {DRIVE_DIR}")
print(f"🎯 Thư mục Target_samples: {TARGET_BASE}")


## 3. Khởi Tạo Mô Hình Mask2Former Swin-S Chuẩn Bài Báo GenEval

In [ ]:
# @title 📦 Nạp mô hình Mask2Former Swin-S & CLIP (Chuẩn GenEval gốc)

# 1. Tải metadata 553 prompts chính thức của GenEval
meta_url = "https://raw.githubusercontent.com/leekwanreal/RS-LiDAR/main/prompt_files/geneval_metadata.jsonl"
meta_path = "geneval_metadata.jsonl"
if not os.path.exists(meta_path):
    print("⬇️ Đang tải metadata GenEval...")
    !wget -q {meta_url} -O {meta_path}

prompts_meta = []
with open(meta_path, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            prompts_meta.append(json.loads(line))
print(f"✅ Đã tải thành công metadata cho {len(prompts_meta)} prompts GenEval.")

# 2. Nạp mô hình Mask2Former Swin-S COCO
print("\n📦 Đang nạp mô hình Mask2Former Swin-S COCO (facebook/mask2former-swin-small-coco-instance)...\n   (Chính xác mô hình chuẩn của bài báo GenEval, mAP ~52%)")
mask2former_id = "facebook/mask2former-swin-small-coco-instance"
image_processor = AutoImageProcessor.from_pretrained(mask2former_id)
image_processor.size = {'shortest_edge': 800, 'longest_edge': 1333}  # Chuẩn bài báo GenEval / MMDetection COCO gốc (800px)
detector = Mask2FormerForUniversalSegmentation.from_pretrained(mask2former_id).to(device).eval()
id2label = detector.config.id2label

# 3. Nạp mô hình CLIP ViT-B/32 để phân loại màu sắc chính xác
print("📦 Đang nạp mô hình CLIP ViT-B/32 cho kiểm tra thuộc tính màu sắc (color_attr)...\n")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

COLORS = ['red', 'orange', 'yellow', 'green', 'blue', 'purple', 'pink', 'brown', 'black', 'white']
COLOR_TEXT_EMBEDDINGS = {}

def get_color_text_embeddings(classname):
    """Mã hóa văn bản màu sắc chuẩn 3 prompt templates từ GenEval gốc (có cache)"""
    if classname in COLOR_TEXT_EMBEDDINGS:
        return COLOR_TEXT_EMBEDDINGS[classname]
    prompts = []
    for c in COLORS:
        prompts.extend([
            f"a photo of a {c} {classname}",
            f"a photo of a {c}-colored {classname}",
            f"a photo of a {c} object"
        ])
    inputs = clip_processor(text=prompts, return_tensors="pt", padding=True).to(device)
    with torch.inference_mode():
        text_feats = clip_model.get_text_features(**inputs)
        text_feats = text_feats / text_feats.norm(dim=-1, keepdim=True)
        text_feats = text_feats.view(len(COLORS), 3, -1).mean(dim=1)
        text_feats = text_feats / text_feats.norm(dim=-1, keepdim=True)
    COLOR_TEXT_EMBEDDINGS[classname] = text_feats
    return text_feats

def classify_crop_color(img, box, mask, classname):
    """Phân loại màu sắc chuẩn GenEval: Ghép mask lên nền xám trung tính #999"""
    if img is None or img.width <= 0 or img.height <= 0:
        return 'unknown'
    try:
        x1, y1, x2, y2 = int(max(0, box[0])), int(max(0, box[1])), int(min(img.width, box[2])), int(min(img.height, box[3]))
        if (x2 - x1 <= 12 or y2 - y1 <= 12):
            return 'unknown'
        if mask is not None:
            blank = Image.new("RGB", img.size, color="#999")
            comp = Image.composite(img, blank, Image.fromarray((mask > 0).astype(np.uint8) * 255))
            crop_img = comp.crop((x1, y1, x2, y2))
        else:
            crop_img = img.crop((x1, y1, x2, y2))
        
        inputs = clip_processor(images=crop_img, return_tensors="pt").to(device)
        with torch.inference_mode():
            img_feats = clip_model.get_image_features(**inputs)
            img_feats = img_feats / img_feats.norm(dim=-1, keepdim=True)
            text_feats = get_color_text_embeddings(classname)
            logits = img_feats @ text_feats.T
            best_idx = logits.argmax(dim=-1).item()
            return COLORS[best_idx]
    except Exception:
        return 'unknown'

def relative_position(box_a, box_b):
    """Tính vị trí tương quan của A đối với B (chuẩn bài báo GenEval gốc)"""
    boxes = np.array([box_a[:4], box_b[:4]])[:, :4].reshape(2, 2, 2)
    center_a, center_b = boxes.mean(axis=-2)
    dim_a, dim_b = np.abs(np.diff(boxes, axis=-2))[..., 0, :]
    offset = center_a - center_b
    revised_offset = np.maximum(np.abs(offset) - 0.1 * (dim_a + dim_b), 0) * np.sign(offset)
    if np.all(np.abs(revised_offset) < 1e-3):
        return set()
    dx, dy = revised_offset / np.linalg.norm(offset)
    relations = set()
    if dx < -0.5: relations.add("left of")
    if dx > 0.5: relations.add("right of")
    if dy < -0.5: relations.add("above")
    if dy > 0.5: relations.add("below")
    return relations
print("🎉 Toàn bộ hệ thống đánh giá GenEval chuẩn đã sẵn sàng!")


## 4. Cấu Hình 6 Thư Mục Chuẩn & Tiền Kiểm Tra Tồn Tại

In [ ]:
# @title 🔍 Cấu hình đường dẫn cứng cho đúng 6 thư mục & Kiểm tra tồn tại

# 6 thư mục cố định chuẩn xác cần chấm điểm:
TARGET_EXPERIMENTS = [
    # --- SETTING 1: SD v1.5 DDIM-50 (eta=0.0) ---
    {
        "setting": "SD v1.5 DDIM-50",
        "method": "Vanilla LiDAR",
        "folder": "LiDAR_SD15_DPM5_n50_DDIM50_s12.5_lmbda5000_seed100_A100"
    },
    {
        "setting": "SD v1.5 DDIM-50",
        "method": "RS-LiDAR (σ=1.0)",
        "folder": "RSLiDAR_SD15_DPM5_n50_sig1.0_M4_DDIM50_s12.5_lmbda5000_seed100_A100"
    },

    # --- SETTING 2: SD v1.5 DDPM-100 (eta=1.0) ---
    {
        "setting": "SD v1.5 DDPM-100",
        "method": "Vanilla LiDAR",
        "folder": "LiDAR_SD15_DPM5_n50_DDPM100_s12.5_lmbda5000_seed100_A100"
    },
    {
        "setting": "SD v1.5 DDPM-100",
        "method": "RS-LiDAR (σ=1.0)",
        "folder": "RSLiDAR_SD15_DPM5_n50_sig1.0_M4_DDPM100_s12.5_lmbda5000_seed100_A100"
    },

    # --- SETTING 3: SDXL DDPM-100 (eta=1.0) ---
    {
        "setting": "SDXL DDPM-100",
        "method": "Vanilla LiDAR",
        "folder": "LiDAR_SDXL_DMD1_Step1_n100_DDPM100_s8.0_lmbda5000_seed100_A100"
    },
    {
        "setting": "SDXL DDPM-100",
        "method": "RS-LiDAR (σ=1.0)",
        "folder": "RSLiDAR_SDXL_DMD1_Step1_n100_sig1.0_M4_DDPM100_s8.0_lmbda5000_seed100_A100"
    }
]

print("=" * 95)
print("🔍 KIỂM TRA SỰ TỒN TẠI CỦA ĐÚNG 6 THƯ MỤC TRƯỚC KHI CHẤM:")
print("=" * 95)

for i, exp in enumerate(TARGET_EXPERIMENTS, 1):
    exp["path"] = os.path.join(TARGET_BASE, exp["folder"])
    exists = os.path.exists(exp["path"]) and os.path.isdir(exp["path"])
    prompt_count = len(glob.glob(f"{exp['path']}/[0-9]*")) if exists else 0
    exp["prompt_count"] = prompt_count
    
    status_text = "✅ TỒN TẠI" if exists else "❌ KHÔNG TÌM THẤY"
    print(f"{i}. {status_text} | [{exp['setting']}] {exp['method']}:")
    print(f"   📁 {exp['path']} ({prompt_count}/553 prompts)")
    
    # Dừng lại ngay nếu có thư mục nào không tồn tại để người dùng biết chính xác
    assert exists, f"❌ Lỗi: Không tìm thấy thư mục: {exp['path']} trên Drive!"

print("=" * 95)
print("🎉 XÁC NHẬN 100%: CẢ 6 THƯ MỤC ĐỀU TỒN TẠI ĐẦY ĐỦ TRÊN DRIVE! SẴN SÀNG CHẤM ĐIỂM.")
print("=" * 95)


## 5. Chấm Điểm GenEval & Lưu Checkpoint Tức Thì Sau Mỗi Thư Mục

In [ ]:
# @title 🚀 BẮT ĐẦU CHẤM ĐIỂM GENEVAL & LƯU CHECKPOINT TỨC THÌ TỪNG FOLDER

# ==============================================================================
# ⚙️ CẤU HÌNH CHẾ ĐỘ CHẤM (RESCORE MODE):
# - "SDXL_ONLY": CHỈ CHẤM LẠI 2 FOLDER SDXL (bỏ qua 4 folder SD 1.5, giữ nguyên điểm đẹp đã có)
# - "ALL":       Chấm lại toàn bộ 6 folder từ đầu trên chuẩn 800px chính thức của GenEval
# - "RESUME":    Chỉ tiếp tục chấm các folder chưa hoàn thành (.mask2former_rescore_done)
# ==============================================================================
RESCORE_MODE = "ALL"

# Đảm bảo detector luôn chạy đúng chuẩn 800px kể cả khi không chạy lại Cell 6:
image_processor.size = {'shortest_edge': 800, 'longest_edge': 1333}
print(f"📐 [CẤU HÌNH ĐỘ PHÂN GIẢI GENEVAL]: {image_processor.size} (Chuẩn bài báo gốc 800px)\n")

def parse_metric_val(val):
    """Trích xuất giá trị số an toàn từ float, int, str hoặc dict {'mean': ...}"""
    if val is None: return None
    if isinstance(val, dict):
        for k in ["mean", "avg", "val", "value"]:
            if k in val and val[k] is not None:
                try: return float(val[k])
                except (ValueError, TypeError): pass
        for v in val.values():
            try: return float(v)
            except (ValueError, TypeError): pass
    try:
        return float(val)
    except (ValueError, TypeError):
        return None

def extract_metrics(folder_path):
    """Đọc ImageReward, CLIP, HPS v2.1 trực tiếp từ final_metrics.json hoặc results.json"""
    if not folder_path or not os.path.exists(folder_path):
        return {"ir": "N/A", "clip": "N/A", "hps": "N/A", "count": 0}

    ir_val, clip_val, hps_val, count = None, None, None, 0

    # 1. Đọc từ final_metrics.json
    fm_candidates = [
        os.path.join(folder_path, "final_metrics.json"),
        os.path.join(folder_path, "final_metrics_shard_0.json")
    ]
    for fm_path in fm_candidates:
        if os.path.exists(fm_path):
            try:
                with open(fm_path, 'r', encoding='utf-8') as f:
                    fm = json.load(f)
                for k in ["ImageReward", "image_reward", "ir"]:
                    if k in fm: ir_val = parse_metric_val(fm[k]); break
                for k in ["Clip-Score", "CLIP", "clip", "clip_score"]:
                    if k in fm: clip_val = parse_metric_val(fm[k]); break
                for k in ["HumanPreference", "HPS", "hps", "hps_score"]:
                    if k in fm: hps_val = parse_metric_val(fm[k]); break
                count = len(glob.glob(f"{folder_path}/[0-9]*"))
                if ir_val is not None: break
            except Exception:
                pass

    # 2. Dự phòng quét results.json
    if ir_val is None or clip_val is None or hps_val is None:
        r_files = glob.glob(f"{folder_path}/[0-9]*/results.json")
        if r_files:
            irs, clips, hpss = [], [], []
            for rf in r_files:
                try:
                    with open(rf, 'r', encoding='utf-8') as f:
                        d = json.load(f)
                    if ir_val is None:
                        for k in ["image_reward", "ImageReward", "ir"]:
                            if k in d:
                                v = parse_metric_val(d[k])
                                if v is not None: irs.append(v); break
                    if clip_val is None:
                        for k in ["clip_score", "CLIP", "Clip-Score", "clip"]:
                            if k in d:
                                v = parse_metric_val(d[k])
                                if v is not None: clips.append(v); break
                    if hps_val is None:
                        for k in ["hps_score", "HumanPreference", "HPS", "hps"]:
                            if k in d:
                                v = parse_metric_val(d[k])
                                if v is not None: hpss.append(v); break
                except Exception:
                    pass
            if ir_val is None and irs: ir_val = float(np.mean(irs))
            if clip_val is None and clips: clip_val = float(np.mean(clips))
            if hps_val is None and hpss: hps_val = float(np.mean(hpss))
            if count == 0: count = len(r_files)

    return {
        "ir": f"{ir_val:.4f}" if ir_val is not None else "N/A",
        "clip": f"{clip_val:.4f}" if clip_val is not None else "N/A",
        "hps": f"{hps_val:.4f}" if hps_val is not None else "N/A",
        "count": count
    }

def save_master_table_checkpoint():
    """Cập nhật và ghi đè file table2_master_6rows_summary.csv ngay lập tức sau MỖI folder"""
    rows = []
    for exp in TARGET_EXPERIMENTS:
        m = extract_metrics(exp.get("path"))
        ge = exp.get("geneval_score")
        if ge is None and exp.get("path"):
            done_marker = f"{exp['path']}/.mask2former_rescore_done"
            ge_csv = f"{exp['path']}/geneval_summary.csv"
            # Chỉ đọc điểm nếu đã được chấm xong chuẩn xác bằng Mask2Former
            if os.path.exists(done_marker) and os.path.exists(ge_csv):
                try:
                    df_c = pd.read_csv(ge_csv)
                    for _, r in df_c.iterrows():
                        if 'OVERALL' in str(r.iloc[0]).upper():
                            ge = float(r.iloc[2]); break
                except Exception:
                    pass
        
        ge_str = f"{ge:.4f}" if ge is not None else "⏳ Đang chờ..."
        method_name = f"🔥 {exp['method']}" if "RS-LiDAR" in exp["method"] else exp["method"]
        rows.append({
            "Setting (Cấu Hình)": exp["setting"],
            "Phương Pháp": method_name,
            "ImageReward ↑": m["ir"],
            "CLIP-Score ↑": m["clip"],
            "HPS v2.1 ↑": m["hps"],
            "GenEval ↑": ge_str,
            "Số Prompts": f"{m['count']}/553",
            "Thư Mục Dữ Liệu": exp["folder"]
        })
    
    df_master = pd.DataFrame(rows)
    csv_out = f"{TARGET_BASE}/table2_master_6rows_summary.csv"
    df_master.to_csv(csv_out, index=False)
    return df_master, csv_out

def evaluate_geneval_for_folder(target_dir, exp_name):
    """Đánh giá ảnh bằng Mask2Former Swin-S, lưu geneval_summary.csv tại chỗ"""
    assert os.path.exists(target_dir), f"❌ Thư mục không tồn tại: {target_dir}"

    geneval_csv_path = f"{target_dir}/geneval_summary.csv"
    done_marker = f"{target_dir}/.mask2former_rescore_done"

    is_sdxl = "SDXL" in exp_name.upper()
    # 1. Chế độ SDXL_ONLY: Bỏ qua 4 folder SD 1.5 và giữ nguyên kết quả đẹp đã có
    if RESCORE_MODE == "SDXL_ONLY" and not is_sdxl:
        if os.path.exists(geneval_csv_path):
            try:
                df_exist = pd.read_csv(geneval_csv_path)
                for _, r in df_exist.iterrows():
                    if 'OVERALL' in str(r.iloc[0]).upper():
                        score = float(r.iloc[2])
                        print(f"\n⏩ [BỎ QUA - GIỮ NGUYÊN SD 1.5] {exp_name}: GenEval = {score:.4f}")
                        return score
            except Exception:
                pass
        print(f"\n⏩ [BỎ QUA THEO CHẾ ĐỘ SDXL_ONLY] {exp_name}")
        return None

    # 2. Chế độ RESUME: Bỏ qua nếu đã có marker hoàn thành
    if RESCORE_MODE == "RESUME" and os.path.exists(done_marker):
        try:
            df_exist = pd.read_csv(geneval_csv_path)
            for _, r in df_exist.iterrows():
                if 'OVERALL' in str(r.iloc[0]).upper():
                    score = float(r.iloc[2])
                    print(f"\n⚡ [ĐÃ HOÀN THÀNH TRƯỚC ĐÓ] {exp_name}: GenEval = {score:.4f} -> Bỏ qua!")
                    return score
        except Exception:
            pass
    if os.path.exists(geneval_csv_path):
        print(f"\n🔄 Phát hiện file geneval cũ trong {exp_name} -> Tiến hành CHẤM LẠI chuẩn xác bằng Mask2Former Swin-S...")

    print(f"\n" + "="*75)
    print(f"🎯 Đang chấm GenEval (Mask2Former Swin-S): {exp_name}")
    print("   Thanh tiến trình tqdm bắt đầu chạy ngay lập tức...")
    print("="*75)

    task_results = {'single_object': [], 'two_object': [], 'counting': [], 'colors': [], 'position': [], 'color_attr': []}
    scored_prompts_count = 0
    
    # CHẠY TRỰC TIẾP QUA 553 PROMPTS: TQDM HIỆN NGAY LẬP TỨC 0.00s (KHÔNG PHẢI CHỜ QUÉT DRIVE)
    for p_idx in tqdm(range(len(prompts_meta)), desc=f"Chấm: {exp_name[:25]}..."):
        meta = prompts_meta[p_idx]
        tag = meta.get('tag', 'single_object')
        if tag not in task_results: continue

        # Tìm ảnh trực tiếp tại thư mục prompt p_idx (hỗ trợ cả int và 5-digit zero-pad)
        p_dir = f"{target_dir}/{p_idx}"
        if not os.path.exists(p_dir):
            p_dir_alt = f"{target_dir}/{p_idx:05d}"
            if os.path.exists(p_dir_alt):
                p_dir = p_dir_alt
            else:
                continue

        imgs = sorted(glob.glob(f"{p_dir}/samples/*.png")) or [f for f in glob.glob(f"{p_dir}/*.png") if not f.endswith("grid.png")]
        if not imgs:
            continue

        scored_prompts_count += 1
        prompt_scores = []
        for img_path in imgs:
            try:
                img = Image.open(img_path).convert('RGB')
                # Cấu hình chuẩn bài báo GenEval gốc: Cạnh ngắn luôn là 800px (chuẩn MMDetection COCO)
                image_processor.size = {'shortest_edge': 800, 'longest_edge': 1333}
                inputs = image_processor(images=img, return_tensors="pt").to(device)
                with torch.inference_mode():
                    outputs = detector(**inputs)

                # Ngưỡng tin cậy chuẩn bài báo GenEval gốc: 0.9 cho counting, 0.3 cho tất cả các task khác
                conf_thr = 0.9 if tag == 'counting' else 0.3
                results = image_processor.post_process_instance_segmentation(
                    outputs, target_sizes=[img.size[::-1]], threshold=conf_thr
                )[0]
                segmentation = results["segmentation"].detach().cpu().numpy()
                segments_info = results["segments_info"]

                # Sắp xếp các vật thể theo confidence giảm dần (chuẩn bài báo GenEval gốc)
                segments_info = sorted(segments_info, key=lambda s: s.get("score", 0.0), reverse=True)

                detected_objects = []
                for seg in segments_info:
                    c_name = id2label[seg["label_id"]].lower()
                    mask = (segmentation == seg["id"])
                    y_indices, x_indices = np.where(mask)
                    if len(x_indices) > 0 and len(y_indices) > 0:
                        x1, x2 = float(np.min(x_indices)), float(np.max(x_indices))
                        y1, y2 = float(np.min(y_indices)), float(np.max(y_indices))
                        box = [x1, y1, x2, y2, seg.get("score", 1.0)]
                        center_x = (x1 + x2) / 2.0
                        center_y = (y1 + y2) / 2.0

                        pred_color = 'unknown'
                        if (x2 - x1 > 12 and y2 - y1 > 12) and (tag in ['colors', 'color_attr']):
                            pred_color = classify_crop_color(img, box, mask, c_name)

                        detected_objects.append({
                            'class': c_name, 'box': box, 'mask': mask,
                            'center_x': center_x, 'center_y': center_y,
                            'color': pred_color, 'score': seg.get("score", 1.0)
                        })

                success = False
                includes = meta.get('include', [])
                if tag == 'single_object':
                    req_cls = includes[0]['class'].lower()
                    success = any(req_cls in obj['class'] or obj['class'] in req_cls for obj in detected_objects)
                elif tag == 'two_object':
                    req1, req2 = includes[0]['class'].lower(), includes[1]['class'].lower()
                    success = any(req1 in obj['class'] or obj['class'] in req1 for obj in detected_objects) and \
                               any(req2 in obj['class'] or obj['class'] in req2 for obj in detected_objects)
                elif tag == 'counting':
                    req_cls, target_count = includes[0]['class'].lower(), includes[0]['count']
                    found_count = sum(1 for obj in detected_objects if req_cls in obj['class'] or obj['class'] in req_cls)
                    success = (found_count == target_count)
                elif tag == 'colors':
                    req_cls, req_color = includes[0]['class'].lower(), includes[0]['color'].lower()
                    success = any((req_cls in obj['class'] or obj['class'] in req_cls) and (obj['color'] == req_color) for obj in detected_objects)
                elif tag == 'position':
                    req1, req2 = includes[0]['class'].lower(), includes[1]['class'].lower()
                    pos_info = includes[1].get('position', ['right of', 0])
                    expected_rel = pos_info[0]
                    o1_list = [o for o in detected_objects if req1 in o['class'] or o['class'] in req1]
                    o2_list = [o for o in detected_objects if req2 in o['class'] or o['class'] in req2]
                    if o1_list and o2_list:
                        o1, o2 = o1_list[0], o2_list[0]
                        rels = relative_position(o2['box'], o1['box'])
                        success = (expected_rel in rels)
                elif tag == 'color_attr':
                    success = all(any((inc['class'].lower() in obj['class'] or obj['class'] in inc['class'].lower()) and (obj['color'] == inc['color'].lower()) for obj in detected_objects) for inc in includes)
                prompt_scores.append(1.0 if success else 0.0)
            except Exception:
                pass

        if prompt_scores:
            task_results[tag].append(np.mean(prompt_scores))

    if scored_prompts_count == 0:
        print(f"⚠️ CẢNH BÁO: Không tìm thấy ảnh hợp lệ trong: {exp_name}")
        return None

    summary_rows = []
    all_means = []
    for t_name, scores in task_results.items():
        mean_val = np.mean(scores) if scores else 0.0
        if scores: all_means.append(mean_val)
        summary_rows.append({'Nhiệm Vụ (Task)': t_name, 'Số Lượng Prompt': len(scores), 'Độ Chính Xác (Accuracy ↑)': f"{mean_val:.4f}"})

    overall_geneval = float(np.mean(all_means)) if all_means else 0.0
    summary_rows.append({'Nhiệm Vụ (Task)': '🔥 OVERALL GENEVAL BENCHMARK (MASK2FORMER)', 'Số Lượng Prompt': sum(len(s) for s in task_results.values()), 'Độ Chính Xác (Accuracy ↑)': f"{overall_geneval:.4f}"})
    
    df_geneval = pd.DataFrame(summary_rows)
    df_geneval.to_csv(geneval_csv_path, index=False)
    df_geneval.to_csv(f"{target_dir}/geneval_mask2former_summary.csv", index=False)

    # ĐÁNH DẤU HOÀN THÀNH BẰNG FILE MARKER
    with open(done_marker, 'w', encoding='utf-8') as f:
        json.dump({
            "completed_at": time.ctime(),
            "evaluator": "facebook/mask2former-swin-small-coco-instance",
            "geneval_score": overall_geneval,
            "prompts_evaluated": scored_prompts_count
        }, f, indent=2)

    print(f"💾 [ĐÃ LƯU CHECKPOINT MASK2FORMER TẠI CHỖ]: {geneval_csv_path}")
    print(f"⭐ ĐIỂM OVERALL GENEVAL (MASK2FORMER): {overall_geneval:.4f}")
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        
    return overall_geneval

# ==============================================================================
# 🔄 CHẠY VÒNG LẶP TUẦN TỰ & CẬP NHẬT FILE TỔNG HỢP NGAY SAU MỖI THỰC NGHIỆM
# ==============================================================================
for i, exp in enumerate(TARGET_EXPERIMENTS, 1):
    print(f"\n▶️ [{i}/6] TIẾN HÀNH CHẤM: {exp['setting']} - {exp['method']}")
    exp["geneval_score"] = evaluate_geneval_for_folder(exp["path"], exp["folder"])
    
    # GHI ĐÈ FILE MASTER CSV TRÊN GOOGLE DRIVE NGAY LẬP TỨC
    current_df, master_csv_path = save_master_table_checkpoint()
    print(f"💾 [CẬP NHẬT MASTER CSV TRÊN DRIVE]: {master_csv_path}")
    display(current_df)

print("\n" + "="*95)
print("🎉 TẤT CẢ 6 THỰC NGHIỆM ĐÃ HOÀN TẤT VÀ ĐƯỢC LƯU CHECKPOINT AN TOÀN TRÊN GOOGLE DRIVE!")
print("="*95)


## 6. Bảng Tổng Hợp Master 6 Dòng & Phân Tích Mức Tăng Trưởng (Δ Gain)

In [ ]:
# @title 📈 BẢNG PHÂN TÍCH MỨC TĂNG TRƯỞNG (Δ GAIN) RS-LiDAR vs VANILLA LiDAR

master_csv_path = f"{TARGET_BASE}/table2_master_6rows_summary.csv"
assert os.path.exists(master_csv_path), f"❌ Chưa tìm thấy file: {master_csv_path}"

final_6rows_df = pd.read_csv(master_csv_path)

print("="*110)
print("📊 BẢNG KẾT QUẢ KHOA HỌC CHUẨN 6 DÒNG (3 SETTINGS x 2 METHODS)")
print("   Mô hình đánh giá GenEval: Mask2Former Swin-S COCO (Threshold 0.5) + CLIP ViT-B/32")
print("="*110)
display(final_6rows_df)

print("\n" + "="*110)
print("📈 PHÂN TÍCH MỨC TĂNG TRƯỞNG (Δ GAIN) CỦA RS-LiDAR SO VỚI VANILLA LIDAR:")
print("="*110)

for s_idx in [0, 2, 4]:
    row_lidar = final_6rows_df.iloc[s_idx]
    row_rslidar = final_6rows_df.iloc[s_idx + 1]
    setting_name = row_lidar["Setting (Cấu Hình)"]
    
    try:
        v_ir_base = float(row_lidar["ImageReward ↑"])
        v_ir_rs = float(row_rslidar["ImageReward ↑"])
        delta_ir = v_ir_rs - v_ir_base
        pct_ir = (delta_ir / abs(v_ir_base)) * 100 if abs(v_ir_base) > 1e-6 else 0.0
        
        delta_clip = float(row_rslidar["CLIP-Score ↑"]) - float(row_lidar["CLIP-Score ↑"])
        delta_hps = float(row_rslidar["HPS v2.1 ↑"]) - float(row_lidar["HPS v2.1 ↑"])
        delta_ge = float(row_rslidar["GenEval ↑"]) - float(row_lidar["GenEval ↑"])
        
        print(f"🔹 [{setting_name}]:")
        print(f"   • ImageReward: {delta_ir:+.4f} ({pct_ir:+.2f}%)")
        print(f"   • GenEval:     {delta_ge:+.4f}")
        print(f"   • CLIP-Score:  {delta_clip:+.4f}")
        print(f"   • HPS v2.1:    {delta_hps:+.4f}")
    except Exception as e:
        print(f"🔹 [{setting_name}]: Chưa đủ dữ liệu số để tính delta ({e})")
    print("-" * 70)
